In [1]:
# load a .pt textual inversion file and show / manipulate it
import torch
import numpy as np
import os

1. load a PyTorch tensor from a TI file in the 'textual_inversions' directory. load onto the CPU.
2. print the keys, which is a dictionary, then print the key/value pairs.
3. extract the tensor with the key '*' from the dict under the 'string_to_param' key.
4. convert the tensor to a NumPy array and detach from the GPU. All subsequent ops on NumPy array.
5. show number of vectors and shape of tensor.

In [ ]:
filename = 'TI_Tron_original' # exclude the .pt extension

data = torch.load('textual_inversions/'+filename+'.pt', map_location='cpu')
print('TI loaded successfully from ',filename+'.pt')
print('-----------------------------')
print(data.keys())
print('-----------------------------')
#for each key, print the data
for key in data.keys():
    print('key: ',key,' data[key]: ',data[key])

# Extract the tensor associated with the key '*'
tensor = data['string_to_param']['*']

# Convert tensor to a NumPy array. detach tensor from the GPU
# all ops will be done on the numpy array
np_array = tensor.cpu().detach().numpy()
numvectors = np_array.shape[0] # number of vectors
print('\nNumber of vectors: ',numvectors,'    Shape of the tensor: ',np_array.shape)

#create a copy of the numpy array
np_rolled = np_array.copy()

In [ ]:
# ROLLING
#########

roll_amount = int(input("ROLLING: Enter eg '3' (can be +ve or -ve) : "))

#roll_amount *each row* of the numpy array *separately*
for i in range(numvectors):
    np_rolled[i] = np.roll(np_rolled[i], roll_amount)

# SAVING
########

tensor = torch.tensor(np_rolled, device='cuda:0', requires_grad=True) 
print(tensor.shape)
data['string_to_param']['*'] = tensor  # store the tensor back in the data dictionary

# Add prefix for saving rolled file
filename = filename + "_roll" + str(roll_amount) + ".pt"

directory = "textual_inversions"
if not os.path.exists(directory):
    os.makedirs(directory)

# Save file
filepath = os.path.join(directory, filename)
torch.save(data, filepath)
print(f"The file '{filename}' has been saved to the '{directory}' directory.")
